In [ ]:
import torch.nn as nn
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        # self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        '''
        edge_index : 엣지 정보 (src, dst)의 집합.
        node_features: node별 이전 layer에서 생성된 벡터 정보가 담긴 matrix (H^(l-1)) (|V| * d)
        '''
        src, dst = edge_index # src : user, dst : movie
        deg = torch.zeros(node_features.size(0))

        deg.index_add_(0, src, torch.ones_like(src))
        deg.index_add_(0, dst, torch.ones_like(dst))

        norm = 1 / torch.sqrt(deg[src]) * torch.sqrt(deg[dst])

        src_messages = self.W1(node_feature[src]) + self.W2(node_feature[src] * node_feature[dst]) # scr * dim
        src_messages *= norm
        dst_messages = self.W1(node_feature[dst]) + self.W2(node_feature[dst] * node_feature[src])
        dst_messages *= norm

        aggregates = torch.zero_likes(node_feature)
        aggregates.index_add_(0, src, src_messages)
        aggregates.index_add_(0, dst, dst_messages)

        aggregates[:user_num] = self.W1(aggregates[:user_num])
        aggregates[user_num:] = self.W1(aggregates[user_num:])
        
temp = torch.ones(3)
torch.sqrt(temp)